In [25]:
from pyvi import ViTokenizer, ViPosTagger # thư viện NLP tiếng Việt
from tqdm import tqdm
import numpy as np


In [28]:
import pdfplumber
from pathlib import Path
import re
from pyvi.ViTokenizer import tokenize
from rich.progress import track

# Hàm tiền xử lý: loại bỏ ký tự không phải chữ và số, chuyển thành chữ thường và tách từ
def simple_preprocess(text):
    text = re.sub(r'\W+', ' ', text)  # Loại bỏ ký tự không phải chữ và số
    return text.lower().split()  # Chuyển thành chữ thường và tách từ

# Hàm để lấy dữ liệu từ thư mục chứa các tệp PDF
def get_data(folder_path):
    X = []  # Danh sách chứa dữ liệu (từ đã tiền xử lý)
    y = []  # Danh sách chứa nhãn (tên thư mục)

    # Lấy danh sách các thư mục trong folder_path
    dirs = Path(folder_path).iterdir()

    # Duyệt qua các thư mục
    for path in track(dirs, description="Processing directories..."):
        # Lấy danh sách các tệp trong mỗi thư mục
        file_paths = path.iterdir()

        # Duyệt qua các tệp trong thư mục
        for file_path in file_paths:
            if file_path.suffix.lower() == '.pdf':  # Kiểm tra nếu là tệp PDF
                # Đọc nội dung tệp PDF
                with pdfplumber.open(file_path) as pdf:
                    lines = ""
                    for page in pdf.pages:  # Duyệt qua các trang trong tệp PDF
                        lines += page.extract_text()  # Trích xuất văn bản từ trang

                    # Tiền xử lý văn bản: kết hợp các dòng, loại bỏ ký tự không phải chữ và số
                    lines = ' '.join(lines.splitlines())
                    lines = simple_preprocess(lines)  # Tiền xử lý (tách từ, chuyển thành chữ thường)
                    lines = ' '.join(lines)
                    lines = tokenize(lines)  # Sử dụng pyvi để tokenize

                    X.append(lines)  # Thêm dữ liệu vào danh sách
                    y.append(path.name)  # Lấy tên thư mục làm nhãn
    return X, y

# Ví dụ đường dẫn thư mục dữ liệu
data_folder = r'D:\thaythedoanpj1\data'  # Thay 'path_to_your_data_folder' bằng đường dẫn thư mục của bạn

# Gọi hàm để lấy dữ liệu
X_data, y_data = get_data(data_folder)

# Hiển thị kết quả
print("\nX_data:", X_data)
print("y_data:", y_data)


KeyboardInterrupt: 

In [ ]:
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

# Chuyển đổi X_data thành vector sử dụng TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)  # Chỉ lấy tối đa 5000 đặc trưng (từ)
X_tfidf = vectorizer.fit_transform(X_data)

# Chuyển đổi kết quả thành mảng NumPy (dạng ma trận)
X_tfidf = X_tfidf.toarray()

# Kiểm tra kích thước của ma trận vector hóa
print(X_tfidf.shape)

# Lưu ma trận vector hóa vào tệp .npy
np.save('X_tfidf.npy', X_tfidf)

# Lưu vectorizer vào tệp .pkl
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Lưu dữ liệu theo các chủ đề vào các tệp riêng biệt
# Tạo thư mục lưu dữ liệu nếu chưa có
output_dir = 'output_data'
os.makedirs(output_dir, exist_ok=True)

# Lưu dữ liệu cho từng chủ đề (nhãn)
for label in set(y_data):  # set(y_data) sẽ loại bỏ trùng lặp nhãn
    # Tạo thư mục cho chủ đề nếu chưa có
    topic_dir = os.path.join(output_dir, label)
    os.makedirs(topic_dir, exist_ok=True)

    # Lọc dữ liệu theo chủ đề
    topic_data = [X_tfidf[i] for i in range(len(y_data)) if y_data[i] == label]

    # Lưu dữ liệu của chủ đề vào tệp .npy riêng biệt
    np.save(os.path.join(topic_dir, f'{label}_X_tfidf.npy'), topic_data)

print("Dữ liệu đã được lưu theo chủ đề vào các thư mục riêng biệt.")
